# Synthetic PBR Materials: explore and build a baseline

Dataset: [dianbrown/synthetic-pbr-materials](https://www.kaggle.com/datasets/dianbrown/synthetic-pbr-materials) · CC0 · 3,780 images · seven classes.

Attach this dataset with **Add Input**, then **Run All**. CPU is sufficient;
no internet, GPU, pretrained weights, or repository checkout is needed.
This notebook supports extracted files and a dataset ZIP. Locally, set
`MATERIAL_DATASET_PATH` to an extracted root or ZIP before starting the kernel.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import hashlib
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, ConfusionMatrixDisplay

SEED = 62642
LABELS = ["ceramic", "concrete", "fabric", "leather", "metal", "stone", "wood"]

## Locate the dataset and read its primary manifest

In [ ]:
override = os.environ.get("MATERIAL_DATASET_PATH")
search = Path(override) if override else Path("/kaggle/input")
root, archive, prefix = None, None, ""
if search.is_dir() and (search / "metadata/manifest.csv").is_file():
    root = search
elif search.is_dir():
    candidates = [p.parent.parent for p in search.rglob("metadata/manifest.csv")
                  if (p.parent.parent / "images").is_dir()]
    if len(candidates) > 1:
        raise ValueError("Multiple matching datasets: set MATERIAL_DATASET_PATH explicitly.")
    if candidates:
        root = candidates[0]
if root is None:
    matches = []
    paths = [search] if search.is_file() else sorted(search.rglob("*.zip"))
    for path in paths:
        with ZipFile(path) as candidate:
            for name in candidate.namelist():
                if name.endswith("metadata/manifest.csv"):
                    base = name[:-len("metadata/manifest.csv")]
                    if any(n.startswith(base + "images/development/") for n in candidate.namelist()):
                        matches.append((path, base))
    if len(matches) != 1:
        raise FileNotFoundError("Attach Synthetic PBR Materials using Add Input, or set MATERIAL_DATASET_PATH to its root/ZIP.")
    archive = ZipFile(matches[0][0])
    prefix = matches[0][1]

def read_file(relative):
    return (root / relative).read_bytes() if root else archive.read(prefix + relative)

def load_image(relative):
    with Image.open(BytesIO(read_file(relative))) as image:
        return image.convert("RGB")

manifest = pd.read_csv(BytesIO(read_file("metadata/manifest.csv")), dtype=str)
assert len(manifest) == 3780 and manifest["image_path"].is_unique
assert set(manifest["class"]) == set(LABELS)
assert manifest.groupby("split").size().to_dict() == {"development": 3150, "test": 630}
assert manifest.groupby("material_id").size().eq(30).all()
assert manifest["source_group"].nunique() == 122
print("Loaded", len(manifest), "images from", root or matches[0][0].name)
display(manifest.head(3))

## Class balance and representative development images

There are 450 development and 90 test images per class. We display only
development examples here; the test images remain outside this demonstration.


In [ ]:
counts = pd.crosstab(manifest["class"], manifest["split"]).reindex(LABELS)
display(counts)
counts.plot.bar(figsize=(10, 3), color=["#439cb5", "#dda45f"], rot=0)
plt.ylabel("Images")
plt.title("Current partitions by class")
plt.tight_layout()
plt.show()

development = manifest.loc[manifest["split"].eq("development")].reset_index(drop=True)
test = manifest.loc[manifest["split"].eq("test")].copy()
fig, axes = plt.subplots(2, 7, figsize=(16, 5))
for col, label in enumerate(LABELS):
    subset = development.loc[development["class"].eq(label)]
    selected = subset.drop_duplicates("source_group").sample(2, random_state=SEED)
    for ax, (_, row) in zip(axes[:, col], selected.iterrows()):
        ax.imshow(load_image(row["image_path"]))
        ax.set_title(label + "\n" + row["material_id"], fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

## Verify a sample of packaged images

`sha256` checks the packaged PNG. `source_sha256` refers to the original PNG,
whose ancillary metadata was removed before publication. The bytes differ
while decoded pixels remain unchanged. A full audit can check all rows.


In [ ]:
for _, row in development.sample(7, random_state=SEED).iterrows():
    payload = read_file(row["image_path"])
    assert hashlib.sha256(payload).hexdigest() == row["sha256"]
    image = load_image(row["image_path"])
    assert image.size == (512, 512)
    pixel_bytes = b"RGBA:512:512:" + image.convert("RGBA").tobytes()
    assert hashlib.sha256(pixel_bytes).hexdigest() == row["pixel_sha256"]
print("7 sampled development images passed byte, pixel, and dimension checks.")

## Make a source-group validation split

`original_split` is lineage only: historical train/val/test images all belong
to development now. Split using `source_group` rather than random image rows.
This example uses the first fold of a fixed five-fold stratified group split.
It is a tutorial split, not a reproduction of the study's nested evaluation.


In [ ]:
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, validation_idx = next(splitter.split(development, development["class"], development["source_group"]))
train_groups = set(development.iloc[train_idx]["source_group"])
validation_groups = set(development.iloc[validation_idx]["source_group"])
assert train_groups.isdisjoint(validation_groups)
assert set(development["source_group"]).isdisjoint(set(test["source_group"]))
assert set(development.iloc[train_idx]["class"]) == set(LABELS)
assert set(development.iloc[validation_idx]["class"]) == set(LABELS)
development["tutorial_split"] = "train"
development.loc[validation_idx, "tutorial_split"] = "validation"
display(pd.crosstab(development["class"], development["tutorial_split"]).reindex(LABELS))
print(len(train_groups), "training groups;", len(validation_groups), "validation groups; no overlap.")

## A small CPU baseline

Extract 16-bin RGB histograms and channel means/standard deviations from 64 × 64
thumbnails. Fit a scaler and logistic regression on training groups only. This
simple appearance baseline is intended to demonstrate the workflow; it is not
the project's trained model or its reported benchmark. Background and lighting
can influence these features. No test predictions are made.


In [ ]:
def appearance_features(path):
    image = load_image(path).resize((64, 64), Image.Resampling.BILINEAR)
    array = np.asarray(image, dtype=np.float32) / 255.0
    histograms = [np.histogram(array[:, :, channel], bins=16, range=(0, 1))[0] / 4096
                  for channel in range(3)]
    return np.concatenate(histograms + [array.mean(axis=(0, 1)), array.std(axis=(0, 1))])

features = np.stack([appearance_features(path) for path in development["image_path"]])
labels = development["class"].to_numpy()
baseline = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000, random_state=SEED))
baseline.fit(features[train_idx], labels[train_idx])
predicted = baseline.predict(features[validation_idx])
print(f"Development validation accuracy: {accuracy_score(labels[validation_idx], predicted):.3f}")
print(f"Development validation balanced accuracy: {balanced_accuracy_score(labels[validation_idx], predicted):.3f}")
print(classification_report(labels[validation_idx], predicted, labels=LABELS, zero_division=0))
ConfusionMatrixDisplay.from_predictions(labels[validation_idx], predicted, labels=LABELS,
                                       xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title("Tutorial baseline: development validation only")
plt.tight_layout()
plt.show()

## Save the tutorial split and predictions

Outputs go to the notebook's working directory. The source images and original
manifest remain unchanged. Retain the group split when trying other features.


In [ ]:
development[["image_path", "class", "material_id", "source_group", "tutorial_split"]].to_csv("tutorial_group_split.csv", index=False)
predictions = development.iloc[validation_idx][["image_path", "class", "source_group"]].copy()
predictions["predicted_class"] = predicted
predictions.to_csv("tutorial_validation_predictions.csv", index=False)
print("Saved tutorial_group_split.csv and tutorial_validation_predictions.csv")
if archive is not None:
    archive.close()

## Interpreting results

The study's 630-image test has already been evaluated. Do not use it for repeated
model selection and then describe the result as untouched confirmation. Keep
tuning inside development; new confirmatory claims require a new frozen holdout.
Known source groups do not establish complete physical independence. Multiple
views are correlated, and image-level counts overstate the number of independent
materials. Renderer/taxonomy differences and possible pretraining overlap limit
generalization. These are synthetic appearance labels, not measured composition
or evidence of performance on real photographs.

Credits: Dian Brown / Material Studio; surface assets by their credited Poly
Haven and ambientCG authors. See the dataset's provenance records and CC0 terms.
